# HNC dataset inspection (Dementia + MDD)

Self-contained notebook to inspect the structure of the private HDF5 +
`.pkl` dataset pairs. Only needs `h5py`, `numpy`, and `pandas` installed —
you can run this **before** installing the full pipeline requirements.

**Goal:** capture enough info that the loader (`pipeline/hnc_dataset.py`)
can be tuned for your data without me needing direct access. Run cells
top-to-bottom and paste the full output back to me. Three things matter
most:

1. **Channel count + channel name order** (HDF5 has shape `(N, n_ch, T)`
   but no channel labels — these come from elsewhere).
2. **Voltage scale** (volts vs µV — picked up from sample stats).
3. **`.pkl` schema** — exact column names, dtypes, label encoding, ID format.

## 0. Set the file paths

Edit the four absolute paths below to match where the data lives on your server.

In [ ]:
DEMENTIA_H5  = '/absolute/path/to/Dementia_paper_dataset_rawData.hdf5'
DEMENTIA_PKL = '/absolute/path/to/Dementia_paper_dataset_info.pkl'
MDD_H5       = '/absolute/path/to/MDD_paper_dataset_rawData.hdf5'
MDD_PKL      = '/absolute/path/to/MDD_paper_dataset_info.pkl'

import os
for p in [DEMENTIA_H5, DEMENTIA_PKL, MDD_H5, MDD_PKL]:
    print(p, '->', 'OK' if os.path.exists(p) else '*** MISSING ***')

## 1. Verify the three deps are installed

If any of these fails, run `!pip install h5py pandas numpy` in a cell first.

In [ ]:
import h5py
import numpy as np
import pandas as pd
print('h5py', h5py.__version__)
print('numpy', np.__version__)
print('pandas', pd.__version__)

## 2. HDF5 structure walker

Recursively prints groups, datasets, shapes, dtypes, attributes, and sample
stats. Output is capped so it stays paste-friendly even if there are many
subjects.

In [ ]:
def _short(v, limit=160):
    s = repr(v)
    return s if len(s) <= limit else s[:limit] + f'... (+{len(s)-limit} chars)'

def _print_attrs(obj, indent):
    if len(obj.attrs) == 0:
        return
    print(f'{indent}attrs:')
    for k in obj.attrs.keys():
        try:
            v = obj.attrs[k]
        except Exception as e:
            print(f'{indent}  {k} = <unreadable: {e}>')
            continue
        if isinstance(v, np.ndarray):
            print(f'{indent}  {k}: ndarray shape={v.shape} dtype={v.dtype} preview={_short(v.tolist()[:20] if v.ndim else v.item())}')
        elif isinstance(v, (bytes, np.bytes_)):
            print(f'{indent}  {k}: bytes {_short(v.decode("utf-8", errors="replace"))}')
        else:
            print(f'{indent}  {k} = {_short(v)}')

def _print_dataset(name, ds, indent):
    print(f'{indent}Dataset {name!r}: shape={ds.shape} dtype={ds.dtype} compression={ds.compression}')
    _print_attrs(ds, indent + '  ')
    try:
        if ds.size == 0:
            return
        if ds.dtype.kind in ('i', 'u', 'f'):
            if ds.ndim >= 2:
                slc = tuple(slice(0, min(s, 32 if i == ds.ndim - 1 else 4))
                            for i, s in enumerate(ds.shape))
                sample = np.asarray(ds[slc], dtype=np.float64)
                print(f'{indent}  sample {slc}: min={sample.min():.4g} max={sample.max():.4g} '
                      f'mean={sample.mean():.4g} std={sample.std():.4g}')
                # First 5 values of channel 0 of subject 0 (helps spot units)
                first5 = np.asarray(ds[(0,) * (ds.ndim - 1) + (slice(0, 5),)], dtype=np.float64)
                print(f'{indent}  first 5 samples of subject0/channel0: {first5.tolist()}')
            elif ds.ndim == 1 and ds.shape[0] <= 5000:
                arr = ds[:]
                uniq, counts = np.unique(arr, return_counts=True)
                if len(uniq) <= 30:
                    print(f'{indent}  unique values ({len(uniq)}): {list(zip(uniq.tolist(), counts.tolist()))}')
                else:
                    print(f'{indent}  unique={len(uniq)}, min={arr.min()} max={arr.max()}')
        elif ds.dtype.kind in ('S', 'O', 'U'):
            n = min(5, ds.shape[0] if ds.ndim else 1)
            preview = ds[:n] if ds.ndim else ds[()]
            decoded = []
            for x in np.atleast_1d(preview):
                if isinstance(x, (bytes, np.bytes_)):
                    decoded.append(x.decode('utf-8', errors='replace'))
                else:
                    decoded.append(str(x))
            print(f'{indent}  first {len(decoded)}: {_short(decoded)}')
    except Exception as e:
        print(f'{indent}  [preview failed: {e}]')

def walk_hdf5(path, max_children_per_group=20):
    print(f'=== {path} ===')
    with h5py.File(path, 'r') as f:
        print(f'File-level attrs ({len(f.attrs)}):')
        _print_attrs(f, '  ')
        print()
        def _walk(name, obj, depth):
            indent = '  ' * depth
            if isinstance(obj, h5py.Group):
                print(f'{indent}Group {name!r} ({len(obj)} children)')
                _print_attrs(obj, indent + '  ')
                children = sorted(obj.keys())
                for c in children[:max_children_per_group]:
                    _walk(c, obj[c], depth + 1)
                if len(children) > max_children_per_group:
                    print(f'{indent}  ... ({len(children) - max_children_per_group} more children not shown)')
            elif isinstance(obj, h5py.Dataset):
                _print_dataset(name, obj, indent)
        for name in sorted(f.keys()):
            _walk(name, f[name], 1)
    print()
    print()

walk_hdf5(DEMENTIA_H5)
walk_hdf5(MDD_H5)

## 3. ~~`.pkl` schema inspector~~ → see §3a/§3b + §8

The original §3 cell tried `pd.read_pickle` directly. **Skip it** — the
HNC `.pkl` files are Fernet-encrypted ciphertext, not pickle data. Use:
- **§3a + §3b** to confirm the encryption format (one-time diagnostic), and
- **§8** to actually load + inspect the DataFrames via `hnc.function_hnc.hnc_load_data`.

## 3a. Diagnose `.pkl` format (run this if §3 fails)

The `.pkl` files in some HNC distributions are not raw `pickle.dump` output —
they may be **joblib** (compressed pickle), **HDF5 with a `.pkl` extension**,
or a sequence of **multiple pickle objects**. The cells below identify the
real format from the magic number, then try every reasonable loader and
report which one succeeds.

If §3 (the simple `pd.read_pickle` path) raises `UnpicklingError: pickle
data was truncated`, run §3a + §3b first to figure out what loader to use,
then re-run §3 (or just use the dict returned by §3b).

In [ ]:
# --- §3a: identify the real file format from magic number ---
import os

def diagnose_pkl(path):
    print(f'\n=== {path} ===')
    if not os.path.exists(path):
        print('  *** missing ***'); return None
    sz = os.path.getsize(path)
    print(f'  size: {sz:,} bytes ({sz/1e6:.2f} MB)')
    with open(path, 'rb') as f:
        head = f.read(32)
    print(f'  first 32 bytes (hex): {head.hex()}')
    print(f'  first 32 bytes (raw): {head!r}')

    if head.startswith(b'\x89HDF\r\n\x1a\n'):
        print('  -> HDF5 magic detected (file is HDF5 with .pkl extension)')
        return 'hdf5'
    if head[:2] == b'\x1f\x8b':
        print('  -> gzip magic detected (likely joblib with compress=)')
        return 'gzip'
    if head[:4] == b'PK\x03\x04':
        print('  -> ZIP magic detected (joblib zipped, or zipfile container)')
        return 'zip'
    if head[:3] == b'BZh':
        print('  -> bzip2 magic detected (joblib with compress="bz2")')
        return 'bz2'
    if head[:6] == b'\xfd7zXZ\x00':
        print('  -> xz/lzma magic detected (joblib with compress="xz")')
        return 'xz'
    if head[:4] == b'ZF\x07\x01':
        print('  -> joblib uncompressed-with-numpy-arrays magic detected')
        return 'joblib_zf'
    if head.startswith(b'\x80'):
        proto = head[1] if len(head) > 1 else None
        print(f'  -> raw pickle stream (protocol {proto})')
        return 'pickle'
    print('  -> unknown header — paste the raw bytes back to me')
    return 'unknown'

fmt_dem = diagnose_pkl(DEMENTIA_PKL)
fmt_mdd = diagnose_pkl(MDD_PKL)

## 3b. Try every loader and report which one wins

Tries `joblib.load`, `h5py.File`, multi-pickle-stream, and
`pd.read_pickle(compression='infer')` in order. The first one that succeeds
prints the dict keys / DataFrame schema. If **all** loaders fail, the file
is genuinely truncated and the data owner needs to re-share it.

In [ ]:
# --- §3b: try every reasonable loader ---
import os, pickle

def _summarize(obj, indent='    '):
    print(f'{indent}type: {type(obj).__name__}')
    if isinstance(obj, dict):
        print(f'{indent}keys: {list(obj.keys())}')
        for k, v in list(obj.items())[:5]:
            print(f'{indent}  [{k!r}]: type={type(v).__name__}', end='')
            if hasattr(v, 'shape'):  print(f' shape={v.shape}', end='')
            if hasattr(v, 'columns'): print(f' columns={list(v.columns)[:8]}...' if len(v.columns) > 8 else f' columns={list(v.columns)}', end='')
            if hasattr(v, '__len__') and not hasattr(v, 'shape'): print(f' len={len(v)}', end='')
            print()
    elif hasattr(obj, 'shape'):
        print(f'{indent}shape: {obj.shape}')
    elif hasattr(obj, '__len__'):
        print(f'{indent}len: {len(obj)}')

def try_load(path):
    print(f'\n=== {path} ===')
    if not os.path.exists(path):
        print('  *** missing ***'); return None

    # 1) joblib (handles compressed pickle files)
    try:
        import joblib
        d = joblib.load(path)
        print('  ✓ joblib.load succeeded')
        _summarize(d)
        return ('joblib', d)
    except Exception as e:
        print(f'  ✗ joblib.load: {type(e).__name__}: {str(e)[:160]}')

    # 2) h5py (file might be HDF5 with .pkl extension)
    try:
        import h5py
        with h5py.File(path, 'r') as f:
            keys = list(f.keys())
            print(f'  ✓ h5py.File succeeded -> top-level keys: {keys}')
            for k in keys:
                obj = f[k]
                if hasattr(obj, 'shape'):
                    print(f'    {k}: dataset shape={obj.shape} dtype={obj.dtype}')
                else:
                    print(f'    {k}: group with children {list(obj.keys())}')
        return ('hdf5', keys)
    except Exception as e:
        print(f'  ✗ h5py.File: {type(e).__name__}: {str(e)[:160]}')

    # 3) multi-pickle (some files contain multiple pickle objects in sequence)
    try:
        objs = []
        with open(path, 'rb') as f:
            while True:
                try:
                    objs.append(pickle.load(f))
                except EOFError:
                    break
        if not objs:
            raise RuntimeError('no objects read')
        print(f'  ✓ multi-pickle stream: read {len(objs)} object(s)')
        for i, o in enumerate(objs):
            print(f'  -- object [{i}] --')
            _summarize(o, indent='      ')
        return ('multi-pickle', objs)
    except Exception as e:
        print(f'  ✗ multi-pickle: {type(e).__name__}: {str(e)[:160]}')

    # 4) pandas with explicit compression detection
    for comp in ('infer', 'gzip', 'bz2', 'xz', 'zip', None):
        try:
            import pandas as pd
            d = pd.read_pickle(path, compression=comp)
            print(f'  ✓ pd.read_pickle(compression={comp!r}) succeeded')
            _summarize(d)
            return ('pandas', d)
        except Exception as e:
            print(f'  ✗ pd.read_pickle(compression={comp!r}): {type(e).__name__}: {str(e)[:120]}')

    print('  *** ALL LOADERS FAILED — file is likely truncated ***')
    return None

result_dem = try_load(DEMENTIA_PKL)
result_mdd = try_load(MDD_PKL)

## 4. ~~Cross-check HDF5 row count vs `.pkl` row count~~ → already confirmed

Originally tried to compare HDF5 dataset sizes against `.pkl` DataFrame
lengths. Now redundant: **§7** already shows the HDF5 splits decrypt to
shapes `(N, 30, T)` and **§8** shows DataFrames have matching `N` rows
(Dementia 153/58/97, MDD 280/120). The loader will use `df.iloc[i]` to
match position-`i` of the DataFrame to subject-`i` of the HDF5 axis-0.

## 5. ~~Channel count + voltage sanity~~ → see §7

The original §5 cell tried to read sample stats directly from h5py — this
fails because the data is encrypted (h5py only sees the giant ciphertext
string). **§7** does the same check correctly via `hnc.function_hnc.hnc_load_data`
and reports both channel count and voltage scale.

## 6. ⚠️ The one thing the data owner has to tell you

Both files contain **30-channel** EEG (confirmed in §7), but neither the
HDF5 attributes nor the info DataFrames carry channel-name metadata
(confirmed in §7 + §8). The loader cannot select the standard 19-channel
10-20 montage without knowing the order.

**Ask the data owner**: *"What are the 30 channel names along axis 1 of
`train/data` in the decrypted EEG arrays, in order? I need a list like
`['Fp1', 'Fp2', 'F3', ...]` matching the channel axis exactly."*

Same channel order is presumably used for both Dementia and MDD (single
clinical recording protocol), but please confirm with them.

Paste their answer below — the loader will pick this up automatically.

In [ ]:
# Paste the channel order from the data owner here.
# Same list works for both datasets unless they tell you otherwise.
# Order MUST match the HDF5 channel axis (axis 1 of the (N, 30, T) arrays).
HNC_CHANNEL_NAMES = None  # e.g. ['Fp1','Fp2','F3','F4','C3','C4','P3','P4', ...]

if HNC_CHANNEL_NAMES is None:
    print('⚠️  HNC_CHANNEL_NAMES not set yet — ask the data owner.')
    print('    The loader will refuse to start without it.')
else:
    print(f'✓ {len(HNC_CHANNEL_NAMES)} channels recorded:')
    for i, ch in enumerate(HNC_CHANNEL_NAMES):
        print(f'  [{i:2d}] {ch}')

## 7. Decrypt with the `hnc` package and inspect the real arrays

Both data files are **Fernet-encrypted blobs** wrapped in HDF5 (the
`gAAAAAB...` magic in §3a). You confirmed the data owner pre-installed an
`hnc` Python package on this server — its `function_hnc.hnc_load_data()` is
the official decryption helper. Per the spec:

```python
from hnc import function_hnc
data_train = function_hnc.hnc_load_data(path_data, 'train/data')
info_train = function_hnc.hnc_load_data(path_info, 'train')
```

The cell below decrypts one split per dataset and inspects:

- the shape, dtype, and value range of the resulting EEG array
- whether the decrypted output carries channel-name metadata anywhere
- the actual columns of the decrypted info DataFrame

If channel names aren't visible after decryption either, you'll have to
ask the data owner for them — but let's see what the helper returns first.

In [ ]:
# --- §7: decrypt with hnc.function_hnc and inspect the real arrays ---
#
# Loads ONLY the test split per dataset (smallest, ~1.3 GB) to avoid OOM
# and explicitly frees the array between datasets. Test alone gives us
# everything we need: shape, dtype, channel count, voltage scale.
import gc
from hnc import function_hnc

def inspect_one_split(h5_path, split, name):
    print(f'\n=== {name} | {h5_path} | {split}/data ===')
    try:
        arr = function_hnc.hnc_load_data(h5_path, f'{split}/data')
    except Exception as e:
        print(f'  hnc_load_data failed: {type(e).__name__}: {e}')
        return

    print(f'  type: {type(arr).__name__}')
    if isinstance(arr, dict):
        print(f'  dict keys: {list(arr.keys())}')
        for k, v in arr.items():
            shape = getattr(v, 'shape', None)
            print(f'    [{k!r}]: type={type(v).__name__}'
                  + (f' shape={shape}' if shape is not None else ''))
        for ch_key in ('channels', 'ch_names', 'channel_names', 'chs'):
            if ch_key in arr:
                print(f'  *** CHANNEL NAMES under key {ch_key!r}: {arr[ch_key]}')
    elif isinstance(arr, tuple):
        print(f'  tuple len: {len(arr)}')
        for i, v in enumerate(arr):
            shape = getattr(v, 'shape', None)
            print(f'    [{i}]: type={type(v).__name__}'
                  + (f' shape={shape}' if shape is not None else f' value={v!r}'[:80]))
    elif isinstance(arr, np.ndarray):
        print(f'  shape: {arr.shape}  dtype: {arr.dtype}')
        if arr.ndim == 3:
            print(f'  n_subjects (axis 0): {arr.shape[0]}')
            print(f'  n_channels (axis 1): {arr.shape[1]}')
            print(f'  n_samples  (axis 2): {arr.shape[2]}')
            sample = arr[0, :, :1000].astype(np.float64)
            absmax = float(np.abs(sample).max())
            print(f'  subject0/all_ch/first1k: '
                  f'min={sample.min():.4g} max={sample.max():.4g} '
                  f'mean={sample.mean():.4g} std={sample.std():.4g}')
            first5 = arr[0, 0, :5].astype(np.float64).tolist()
            print(f'  subject0/channel0 first 5 samples: {first5}')
            # Voltage scale heuristic — typical EEG is ~20-100 µV peak
            if absmax < 1e-3:
                print(f'  -> volts (absmax={absmax:.2g}); loader should ×1e6')
            elif absmax < 1.0:
                print(f'  -> millivolts (absmax={absmax:.2g} → {absmax*1e3:.1f} µV peak); '
                      f'loader should ×1e3')
            elif absmax < 1e3:
                print(f'  -> microvolts (absmax={absmax:.2g}); no scaling needed')
            else:
                print(f'  -> unusually large ({absmax:.2g}); ASK DATA OWNER')

    # Explicitly free the decrypted array before moving on
    del arr
    gc.collect()

# Only the test split per dataset — keeps memory under control
inspect_one_split(DEMENTIA_H5, 'test', 'Dementia')
inspect_one_split(MDD_H5,      'test', 'MDD')

## 8. Decrypt the info `.pkl` files and inspect the DataFrames

Same helper, different second-arg style. Per the spec the call is
`hnc_load_data(path_info, 'train')` (note: no `/data` suffix). This decrypts
one split's metadata DataFrame at a time.

In [ ]:
# --- §8: decrypt the info .pkl with hnc.function_hnc and inspect ---
#
# Note: Python 3.9 doesn't allow backslashes inside f-string expressions, so
# we precompute every value into a plain variable before the f-string.
import gc
from hnc import function_hnc
import pandas as pd

def inspect_info_split(pkl_path, split, name):
    print(f'\n=== {name} | {pkl_path} | split={split!r} ===')
    try:
        df = function_hnc.hnc_load_data(pkl_path, split)
    except Exception as e:
        print(f'  hnc_load_data failed: {type(e).__name__}: {e}')
        return

    print(f'  type: {type(df).__name__}')
    if isinstance(df, pd.DataFrame):
        print(f'  shape: {df.shape}')
        print(f'  columns: {list(df.columns)}')
        print(f'  dtypes:')
        for col, dt in df.dtypes.items():
            print(f'    {col}: {dt}')

        head_str = df.head(5).to_string()
        print(f'  head(5):')
        for line in head_str.split('\n'):
            print(f'    {line}')

        if 'Label' in df.columns:
            label_counts = df['Label'].value_counts(dropna=False).to_dict()
            print(f'  Label value counts: {label_counts}')
        if 'Group' in df.columns:
            group_counts = df['Group'].value_counts(dropna=False).to_dict()
            print(f'  Group value counts: {group_counts}')
        if 'ID' in df.columns:
            id_dtype = df['ID'].dtype
            id_samples = df['ID'].head(5).tolist()
            id_unique = df['ID'].nunique()
            print(f'  ID dtype: {id_dtype}')
            print(f'  ID samples: {id_samples}')
            print(f'  ID unique: {id_unique} / {len(df)}')

        # Hunt for any hidden channel-info column
        for col in df.columns:
            col_lower = col.lower()
            if 'chan' in col_lower or 'electrode' in col_lower or 'montage' in col_lower:
                first_val = df[col].iloc[0]
                print(f'  *** possible channel-info column {col!r}: {first_val!r}')
    elif isinstance(df, dict):
        print(f'  dict keys: {list(df.keys())}')
    else:
        print(f'  repr: {repr(df)[:300]}')

    del df
    gc.collect()

inspect_info_split(DEMENTIA_PKL, 'train', 'Dementia')
inspect_info_split(DEMENTIA_PKL, 'valid', 'Dementia')
inspect_info_split(DEMENTIA_PKL, 'test',  'Dementia')
inspect_info_split(MDD_PKL,      'train', 'MDD')
inspect_info_split(MDD_PKL,      'test',  'MDD')